# Risk definition for residual load

Implements [`.claude/specs/02-Risk-Definition.md`](../../.claude/specs/02-Risk-Definition.md).

`notebooks/01_eda/team-EDA.ipynb` closed with one explicit open question: the two extremes of
`residual_load` are not one phenomenon — they differ in season, hour, day type, trend and
mechanism — and *"we need to decide whether the risk flag should treat them as one target or
two"*. This notebook answers that question and turns the answer into a concrete, reproducible
labelling methodology.

**This is not an EDA notebook.** Every finding it leans on was already established in
`team-EDA.ipynb` and is restated here in one or two lines, never re-derived.

## What this notebook produces

- Three **threshold bases** for each direction, derived from `data/smard.csv` itself with no
  hardcoded MWh figure: a causal `rolling` trailing-window quantile, a physical `zero` crossing
  (low direction only), and a whole-record `static` quantile kept as a reference.
- **Definition 1** — a per-day risk flag with a reason (the triggering hour and its value), under
  two day rules: `any` and a 3-hour persistence rule.
- **Definition 2** — per-hour flags using those same thresholds, and a derived time range per
  flagged day-direction.
- Two exported artifacts, `data/risk_labels_daily.csv` and `data/risk_labels_hourly.csv`, which
  the modeling spec can load directly.

## Conventions

- `time_series` is the main dataframe, `SERIES` its measured columns, `DERIVED` the engineered
  ones — inherited unchanged from [`03.1-setup.md`](../../.claude/specs/03.1-setup.md).
- `YEARS` is computed at run time; no literal calendar year appears in code.
- **Units:** `MWh` for every level and threshold. The dataset is energy data for the whole grid,
  so an hourly reading is treated as energy, not power.
- **Durations, never row counts.** Every duration-based rule below is expressed as a duration
  ("at least 3 hours"), not as a number of observations ("at least 3 rows"), so a later switch to
  SMARD's 15-minute resolution would not silently change the rule's meaning.

---

## 1 Setup

Inherited verbatim from [`03.1-setup.md`](../../.claude/specs/03.1-setup.md), the spec behind
`team-EDA.ipynb`'s setup section: the data-directory resolver, `time_series`, `SERIES`, `DERIVED`,
`YEARS`, `period_mean` / `period_energy`, `style_timeseries`, `DAY_NAMES` and the season mapping.
Nothing here is re-derived — see that spec and `notebooks/01_eda/team-EDA.ipynb` §1 for the
reasoning behind each piece.

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](../API-connection.ipynb) top to bottom.

The data directory is resolved by walking **upward** from the working directory rather than by a
fixed `"../../data/smard.csv"`, so the notebook runs unmodified whether the kernel starts in
`notebooks/03_risk_classification/` or at the repo root.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print(f"Data directory: {DATA}")

### 1.1 Helpers

Copied unchanged from `team-EDA.ipynb` §1.1. `_complete_periods` drops calendar periods the data
does not fully cover — a plain `.resample()` produces fake edge dips. `style_timeseries` requires
an explicit `ylabel`, so no plot can ship without stating its unit.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    A period counts only if it starts not earlier than the first observation and ends not later than the last observation's closing edge.
    Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (
        periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts (half weeks / months, starting or ending a week on thursday).
    A period is considered an artifact if the range is "not fully covered".
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW).
        Deliberately not ``mwh_per_day / 24``: a month containing the spring Daylight-Saving-Time switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so a comparison table needs no second copy of this
        arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state whether it shows MWh, average MW or MWh/day.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

### 1.2 Colour configuration

The subset of `team-EDA.ipynb` §1.2 this notebook needs. `TAIL_COLOR` carries over unchanged, so
the two directions read the same here as they do in the EDA: cool blue for the low/oversupply
tail, hot red for the high/undersupply tail.

In [ ]:
COLORS = {
    "blue":  "#2C6EBA",
    "red":   "#B10F0F",
    "black": "#1C1C1C",
    "gray":  "#AEB8C5",
    "muted": "#707B8C",
}

# Residual-load extremes, wherever they are shown as a pair:
# cool for oversupply (low), hot for undersupply (high).
TAIL_COLOR = {
    "low": COLORS["blue"],
    "high": COLORS["red"],
}

# One line style per threshold basis, used on every threshold plot below.
BASIS_STYLE = {
    "rolling": {"linestyle": "-", "linewidth": 2.0},
    "static": {"linestyle": "--", "linewidth": 1.4},
    "zero": {"linestyle": ":", "linewidth": 1.4},
}

DIRECTION_LABEL = {"high": "High residual load", "low": "Low / negative residual load"}

print(f"{len(TAIL_COLOR)} directions, {len(BASIS_STYLE)} bases configured")

### 1.3 Load and prepare

Dtype assertion guards against the German Excel-CSV conversion silently leaving a column as text.
Everything after this cell uses `time_series`.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
time_series.head(3)

### 1.4 Derived columns, `SERIES` / `DERIVED`, and `YEARS`

`YEARS` is computed from the loaded data and is the only permitted source of year information in
the rest of the notebook. `spans_gap` marks the row *following* a gap in the hourly index — the
record's only gaps are the spring Daylight-Saving-Time switches, where the local hour 02:00 does
not exist.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.5 Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. The closing self-check re-runs these invariants plus a comparison
against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

### 1.6 What this notebook inherits from `team-EDA.ipynb`

Three findings carry real weight here. They are **cited, not re-derived** — no new evidence is
gathered for them in this notebook.

- **§3.7 — the two tails have almost mirror-image calendar signatures.** The low/negative extreme
  is overwhelmingly recent, concentrated in high-solar months, midday hours and weekends (the
  weekly demand minimum lining up with the daily solar maximum); the high extreme is weekday-only,
  spread evenly across years, in winter months, peaking in the evening with a secondary morning
  peak. §3.7 reads the low extreme as *growing and seasonal* and the high extreme as *stable and
  structural*. **This is the direct justification for treating high and low as two independently
  thresholded directions rather than one signed scale.** §3.7 also selected its 1 % tail *by rank*
  and labelled it explicitly a descriptive slice, not a risk definition.
- **§6.3 — the distribution is close to symmetric with a mild negative skew, and 1.26 % of all
  hours are already negative** — a small share, but growing and highly clustered in summer. There
  is no second mode and no sharp cutoff, so every threshold question here is a judgement call
  rather than a boundary the data hands us.
- **§6.5 — the low tail is stretching downward while the high end barely moves.** P10 falls faster
  than the median year on year; P90 hardly moves at all. This notebook's basis comparison exists
  specifically to make that asymmetry visible in threshold terms.

§6.5's **matched calendar window** (1 January to the record's last complete date, applied
identically to every year, derived from the data and never hardcoded) is the only admissible way
to compare years in this notebook — see §2 on why.

### 1.7 The residual-load identity

Before thresholding `residual_load`, confirm what it actually is in this record:

$$\text{residual\_load} = \text{grid\_load} - (\text{wind\_off} + \text{wind\_on} + \text{solar})$$

This is not a data-quality check for its own sake. If the identity holds, then SMARD *defines*
residual load as load minus wind and solar only — which means the project's "wind + solar only"
scope simplification costs **nothing for this target**. Every other generation source (biomass,
coal, hydro, ...) is already outside residual load by construction, not by our choice.

In [ ]:
implied = time_series["grid_load"] - time_series[["wind_off", "wind_on", "solar"]].sum(axis=1)
residual_error = time_series["residual_load"] - implied

abs_err = residual_error.abs()
typical = time_series["residual_load"].abs().median()
exact = (residual_error == 0).mean()
within_1 = (abs_err <= 1).mean()
over_1 = abs_err > 1

print(f"hours checked            : {len(residual_error):,}")
print(f"identity holds exactly   : {exact:6.2%}")
print(f"holds to within 1 MWh    : {within_1:6.2%}")
print(f"max deviation            : {abs_err.max():,.2f} MWh "
      f"({abs_err.max() / typical:.4%} of a typical |residual_load| of {typical:,.0f} MWh)")

if over_1.any():
    print(f"hours deviating > 1 MWh  : {int(over_1.sum())}, confined to "
          f"{residual_error[over_1].index.min():%Y-%m-%d} .. "
          f"{residual_error[over_1].index.max():%Y-%m-%d}")

**The identity holds to within rounding.** It is satisfied exactly for the large majority of
hours and to within 1 MWh for essentially all of them; the handful of larger deviations are
confined to a single short patch of the record and peak at a fraction of a per-mille of a typical
residual-load value — SMARD-side rounding and revision artefacts, not a different definition. At
the scale this notebook thresholds on (tens of thousands of MWh) they are immaterial.

The consequence is the point of the check: SMARD **defines** residual load as grid load minus wind
and solar, so the project's "wind + solar only" scope simplification costs **nothing for this
target**. Biomass, coal, hydro and the rest are outside residual load by construction, not by our
choice.

---

## 2 Defining risk

**Risk, in this project, is a `residual_load` magnitude extreme enough that the TSOs plausibly
need intervention measures to keep the grid balanced** — redispatch, reserve activation,
cross-border exchange, or curtailment.

`residual_load` is what is left of national demand once wind and solar have been subtracted (§1.7).
It is the quantity the remaining dispatchable fleet, storage, imports and exports have to close.
When it is extreme in *either* direction, closing it stops being routine.

That definition splits into **two mechanisms, not one scale**:

| Direction | Mechanism | Plausible intervention |
|---|---|---|
| **High** residual load | Least renewable cover relative to demand — the *Dunkelflaute*-type winter case: high demand, little wind, no solar | Upward redispatch, reserve activation, imports |
| **Low / negative** residual load | Renewable oversupply — wind and solar alone approach or exceed national demand | Downward redispatch, curtailment, exports, negative prices |

### Why two directions and not one signed scale

Because `team-EDA.ipynb` §3.7 already established that **the two extremes have almost mirror-image
calendar signatures** — they are not two ends of one phenomenon:

- The **low/negative** extreme is overwhelmingly **recent**, concentrated in high-solar months, at
  **midday**, and on **weekends** — the weekly demand minimum lining up with the daily solar
  maximum. §3.7 reads it as *growing and seasonal*, a consequence of renewable build-out.
- The **high** extreme is **weekday-only**, **spread evenly across the years** of the record, in
  **winter** months, peaking in the **evening** with a secondary morning peak. §3.7 reads it as
  *stable and structural*.

Different season, different hour, different day type, different mechanism. A single signed
threshold on one scale would have to pretend these are the same event seen from two sides. They
are not, so each direction gets its own threshold, computed independently, throughout this
notebook.

This is the open question `team-EDA.ipynb` closed on — *"we need to decide whether the risk flag
should treat them as one target or two"* — and **this notebook answers: two.** Whether a *model*
should then be trained as one two-sided target or two separate ones is a modelling decision, left
open in §7.

### 2.1 What this data cannot establish

These are **limits on the claim**, not caveats to bury. Each one bounds what the exported label is
allowed to be described as.

**1 — No margin.** The high direction is a *relative* extreme: the highest-residual-load hours
*in this record*. It is **not** a demonstrated approach to a capacity limit. SMARD's region-`DE`
series carry no installed-capacity, plant-availability or cross-border-capacity figures, so
"tight reserve margins" is simply not a claim this notebook can make — we can say an hour is
unusually high for this record, and nothing about how close the fleet came to running out.
(`team-EDA.ipynb` §3.7 uses the phrase "tight reserve margins" in passing; that reading goes
beyond what region-`DE` data supports, and is not adopted here.)

**2 — No regional detail.** The largest real driver of German redispatch is **north–south
transmission congestion** — high northern wind pushing against southern load — which can occur at
a perfectly moderate *national* residual load and is completely invisible in region-`DE` data. The
label produced here is therefore a **national-balance proxy**, and must be described as one
wherever it is presented. A day this label calls quiet may still have required intervention, and
the reverse.

**3 — No intervention record.** Nothing in `data/smard.csv` records whether a TSO actually
intervened on a given day. There is no ground truth here to validate against. The label is a
**plausibility proxy, never a validated outcome**.

### 2.2 What "risk" means here, precisely

This is a **level-based magnitude definition**, not a probability-of-intervention model. This
notebook produces the *candidate label* that a future model would be trained to predict — it does
not estimate any probability, and it fits no model of its own.

### 2.3 A trend claim this notebook does not make

**No text in this notebook asserts that the high tail is trending, in either direction.**
`team-EDA.ipynb` §3.7 reads it as spread across the years rather than trending, and nothing
computed here is entitled to overturn that. Two traps make it easy to believe otherwise:

- **The current year is partial.** The record ends mid-year, so comparing a complete early year
  against the in-progress final one is not a comparison at all. Any year-on-year view of
  `residual_load` would have to use §6.5's **matched calendar window** (1 January to the record's
  last complete date, applied identically to every year, with the end derived from the data rather
  than hardcoded).
- **Even matched, the window discards most of the high tail.** Roughly half of the high 1 % of
  hours falls in October–December, which a matched January-to-current window cuts out entirely.
  The year-to-year variation that survives is several times larger than any slope one could fit
  through eight noisy years.

**This notebook therefore shows no year-on-year view of the tails**, and the case against the
`static` basis in §3 rests on re-fetch instability and non-causality — arguments that do not
depend on any trend claim — rather than on how the tails move over time.

---

## 3 Threshold construction

Every threshold below is **derived from `data/smard.csv` at run time**. No MWh figure is hardcoded
anywhere in this notebook — the record's extent has already changed once (the fetch was widened
back to 2019) and will change again.

Three **bases** are computed, side by side, for comparison:

| Basis | Direction | Definition |
|---|---|---|
| `rolling` | both | Trailing 365-day quantile, using only days strictly **before** the day being labelled |
| `zero` | low only | The physical oversupply boundary, `0` |
| `static` | both | Whole-record quantile — kept as a **reference**, not recommended (§3.4) |

They are **not** presented as equals. §3.4 ranks them.

### 3.1 Rule constants, expressed as durations

Per the convention stated in the header: every duration-based rule is a **duration**, and the
number of observations it corresponds to is *derived* from the record's own resolution. At hourly
resolution "at least 3 hours" is 3 observations; if the record is ever switched to SMARD's
15-minute series, the same line becomes 12 observations with no edit.

The day-completeness rule works the same way — a day must carry at least 23/24 of its expected
observations. That accepts the 23-hour spring-DST days (of which this record has one per year) and
rejects any day left materially short by a future re-fetch.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

WINDOW = pd.Timedelta(days=365)      # trailing history for the rolling basis
PERSISTENCE = pd.Timedelta("3h")     # the `3h` day rule
DAY_COMPLETENESS = 23 / 24           # accepts the spring-DST day, rejects materially short days

MIN_RUN = int(PERSISTENCE / RESOLUTION)
EXPECTED_OBS_PER_DAY = int(pd.Timedelta("1D") / RESOLUTION)
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

residual = time_series["residual_load"]

# One row per local calendar date — the same day boundary DERIVED["date"] uses, not a rolling 24h.
DAY = pd.Series(residual.index.normalize(), index=residual.index)
DAYS = pd.DatetimeIndex(sorted(DAY.unique()))
obs_per_day = residual.groupby(DAY).size().reindex(DAYS, fill_value=0)
day_complete = obs_per_day >= MIN_OBS_PER_DAY

print(f"resolution            : {RESOLUTION}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"persistence rule      : {PERSISTENCE}  ->  {MIN_RUN} consecutive observations")
print(f"day completeness      : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")
print(f"calendar days         : {len(DAYS):,}  ({DAYS.min():%Y-%m-%d} .. {DAYS.max():%Y-%m-%d})")
print(f"observations per day  : {obs_per_day.value_counts().sort_index(ascending=False).to_dict()}")
print(f"days failing the completeness rule : {int((~day_complete).sum())}")

### 3.2 The rolling basis — the recommended one

For each calendar day `D`, the rolling thresholds are the quantiles of `residual_load` over the
trailing **365 days ending the day before `D`** — the window `[D-365, D-1]`.

**`D`'s own data never contributes to its own threshold.** That is what makes this basis *causal*:
the label for a day could have been produced on the morning of that day, which is the only
property that lets a day-ahead model be evaluated honestly against it.

A **full 365 days** of trailing history is required. Days with less get `NaN` — never a threshold
computed from a partial window. The tempting shortcut (allow a threshold once ~90 days exist, so
less of the record is wasted) produces a **seasonally biased** threshold, not merely a noisy one:
a window covering only winter yields a high threshold thousands of MWh above the full-year value,
which is then applied to spring and summer days. That bias is systematic and direction-dependent,
so it cannot be waved through as sampling error.

In [ ]:
def rolling_threshold(series, q, window=WINDOW):
    """Trailing-window quantile per calendar day, using only days strictly before it.

    The value for day D is the rolling quantile evaluated at the last observation of day D-1, so
    the window is exactly [D-365, D-1] and D itself is excluded. Days without a full `window` of
    trailing history are NaN.
    """
    roll = series.rolling(window).quantile(q)
    end_of_day = roll.groupby(series.index.normalize()).last()
    end_of_day.index = end_of_day.index + pd.Timedelta(days=1)  # end of D-1 applies to D
    thr = end_of_day.reindex(DAYS)
    return thr.where(thr.index >= series.index.min().normalize() + window)


def constant_threshold(value):
    """A threshold that is the same on every day (the `static` and `zero` bases)."""
    return pd.Series(float(value), index=DAYS)


def hourly_crossings(series, threshold_by_day, direction):
    """Per-observation crossing flags against each observation's own day's threshold.

    Returns nullable booleans: pd.NA wherever the day's threshold is undefined, so an undefined
    threshold can never masquerade as "not at risk".
    """
    thr = pd.Series(
        threshold_by_day.reindex(series.index.normalize()).to_numpy(), index=series.index
    )
    crossed = (series >= thr) if direction == "high" else (series <= thr)
    return crossed.astype("boolean").mask(thr.isna())


def day_rules(hourly_flag, threshold_by_day):
    """The `any` and `3h` day rules from per-observation crossing flags.

    `any`  — at least one observation crosses.
    `3h`   — the crossing persists for at least PERSISTENCE, consecutively, within the day.

    Consecutive *rows* count as consecutive: on the eight spring-DST days the missing 02:00 makes
    a run spanning it one hour longer in wall-clock terms than in rows. The approximation is noted
    rather than corrected, as 03.6 does for STL.

    Days with an undefined threshold, or materially short of a full day, are pd.NA in both rules
    rather than False.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    any_rule = filled.groupby(DAY).any().reindex(DAYS, fill_value=False)

    run_id = (filled != filled.shift()).cumsum()
    run_len = filled.groupby([DAY, run_id]).transform("size").where(filled, 0)
    persist_rule = (run_len.groupby(DAY).max() >= MIN_RUN).reindex(DAYS, fill_value=False)

    valid = (threshold_by_day.notna() & day_complete).reindex(DAYS, fill_value=False)
    return any_rule.astype("boolean").mask(~valid), persist_rule.astype("boolean").mask(~valid)


_probe = rolling_threshold(residual, 0.99)
print(f"rolling basis defined for {int(_probe.notna().sum()):,} of {len(DAYS):,} days")
print(f"excluded for lack of a full {WINDOW.days}-day history : {int(_probe.isna().sum()):,} days "
      f"({DAYS.min():%Y-%m-%d} .. {_probe.first_valid_index() - pd.Timedelta(days=1):%Y-%m-%d})")

### 3.3 The zero and static bases

**`zero` (low direction only): `low_threshold_zero = 0`.** This is the one basis with a meaning
outside this dataset. Below zero, wind and solar alone exceed national demand, so the surplus must
be exported, curtailed, or paid away at negative prices. It takes no percentile parameter and does
not move when the record is re-fetched.

**The high direction has no equivalent physical anchor in this data.** There is no level of
residual load that region-`DE` data marks as "the fleet runs out here" — that would need
installed-capacity and availability figures SMARD does not carry in these series (§2.1). The
asymmetry between the two directions is therefore **deliberate, not an oversight**: the low
direction gets a physical boundary because one exists; the high direction does not, because none
is observable here.

**`static`: the whole-record quantile**, computed once over the full currently-loaded record.

In [ ]:
P_LEVELS = [0.005, 0.01, 0.02, 0.05]
P_DEFAULT = 0.01  # carried into Definitions 1 and 2 — see §3.5

# Which bases exist per direction. The low direction has one more, because zero only means
# something on that side.
BASES = {"high": ["rolling", "static"], "low": ["rolling", "zero", "static"]}


def build_thresholds(p):
    """Every threshold series, per direction x basis, at percentile level `p`."""
    return {
        ("high", "rolling"): rolling_threshold(residual, 1 - p),
        ("high", "static"): constant_threshold(residual.quantile(1 - p)),
        ("low", "rolling"): rolling_threshold(residual, p),
        ("low", "zero"): constant_threshold(0.0),
        ("low", "static"): constant_threshold(residual.quantile(p)),
    }


THRESHOLDS = build_thresholds(P_DEFAULT)

print(f"At the default level p = {P_DEFAULT:.1%}:")
for (direction, basis), thr in THRESHOLDS.items():
    if basis == "rolling":
        print(f"  {direction:4} {basis:7} : {thr.min():>10,.0f} .. {thr.max():>10,.0f} MWh "
              f"(mean {thr.mean():>9,.0f}, varies by day)")
    else:
        print(f"  {direction:4} {basis:7} : {thr.iloc[0]:>10,.0f} MWh (constant)")

### 3.4 Ranking the bases

Unlike the surrounding EDA specs, this notebook does **not** stay neutral between the three.

#### The static basis has two defects

**1 — It is unstable under re-fetch.** The static threshold is a property of *the record's extent*,
not of the grid. Widening the fetch back to 2019 — which this team has already done once — **raised**
the low threshold and retroactively un-flagged days that were flagged before; extending the end
date pushes it the other way. A label that silently changes when you download more data is not a
label.

**2 — It is not causal.** A day in 2022 is judged against quantiles computed from data including
2025–2026. It cannot be reproduced in a live day-ahead pipeline, because most of its inputs had
not happened yet.

#### The rolling basis has costs too

It is **undefined for the record's first 365 days**, and it **moves as history accumulates**, so
the same absolute MWh level can be "risky" in one period and ordinary in another — identical
physics, different label. That is the price of a stable base rate, and it is the trade this
ranking accepts.

#### The ranking

- **Recommended as label bases: `rolling` (both directions) and `zero` (low direction).**
  Rolling is the only causal basis and the only one that keeps the positive rate comparable across
  a chronological split (§3.6 measures this). Zero is the only one that means anything independent
  of this record.
- **Demoted: `static`.** It is still computed, still exported, and still drawn as a reference line
  on the threshold plots — so the comparison stays visible and the team can overrule this
  judgement — but it is **not recommended as a label basis**, for the two reasons above.

Both directions keep all their bases in the export as parallel, clearly labelled candidate
columns. This ranking is a recommendation for the modelling spec to act on, not a deletion of the
alternatives.

### 3.5 Sensitivity to the percentile level

One row per candidate level, per direction. The static threshold is shown as a reference; the
hour- and day-shares are computed on the **rolling** basis (the recommended one), over the days
where it is defined — the first 365 days are excluded from both numerator and denominator.

The gap between the hour-share and the day-share is the point of the table: **a day is flagged if
any single one of its hours crosses**, so the day-share runs several times the hour-share. The
`3h` column shows how much of that survives a persistence requirement.

In [ ]:
def sensitivity(direction):
    rows = []
    for p in P_LEVELS:
        q = 1 - p if direction == "high" else p
        roll = rolling_threshold(residual, q)
        hourly = hourly_crossings(residual, roll, direction)
        any_rule, persist_rule = day_rules(hourly, roll)
        rows.append({
            "level": f"{p:.1%}",
            "static (MWh)": f"{residual.quantile(q):,.0f}",
            "rolling mean (MWh)": f"{roll.mean():,.0f}",
            "rolling min (MWh)": f"{roll.min():,.0f}",
            "rolling max (MWh)": f"{roll.max():,.0f}",
            "hours flagged": f"{hourly.mean():.2%}",
            "days flagged (any)": f"{any_rule.mean():.2%}",
            "days flagged (3h)": f"{persist_rule.mean():.2%}",
        })
    return pd.DataFrame(rows).set_index("level")


for direction in ("high", "low"):
    print(f"\n{DIRECTION_LABEL[direction]} — sensitivity across candidate levels "
          f"(shares on the rolling basis)")
    display(sensitivity(direction))

**Findings.**

- **The day-share runs four to six times the hour-share** at every level, in both directions. That
  is the `any` rule doing what it says: one crossing hour flags the whole day. The multiple is
  largest at the tightest levels, where crossings are rarest and most isolated.
- **Requiring the crossing to persist for 3 hours removes between a third and nearly half of
  flagged days in the high direction, and consistently less in the low direction.** Low-direction
  events are the more persistent of the two, which fits §3.7's reading of them as a broad midday
  solar trough rather than a sharp evening peak.
- **The high direction's hour-share lands slightly *below* its nominal level** (~0.9 % of hours at
  a nominal 1 %). A threshold built from trailing history is applied to a period whose high tail is
  no more extreme than that history, so slightly fewer hours clear it than the quantile would
  suggest on its own data.
- **The low direction's hour-share lands well *above* nominal** — around 1.7 % of hours at a
  nominal 1 %, nearly double. This is not a bug: it is §6.5's downward stretch made visible.
  Because the low tail keeps falling, a threshold computed from the trailing year is systematically
  too lenient for the year that follows, and more hours breach it than the nominal rate. The low
  direction is chasing a moving target in a way the high direction is not.
- **The static and rolling-mean thresholds diverge sharply in the low direction and barely at all
  in the high direction.** In the high direction the two sit within a few hundred MWh of each
  other at every level. In the low direction the static threshold is far below the rolling mean,
  because the whole-record quantile is dragged down by the recent, much more negative years and
  then applied backwards across a record where those values did not occur.

**1 % is carried into Definitions 1 and 2**, for continuity with the 1 % tail-by-rank framing
`team-EDA.ipynb` §3.7 already used. This is a **recommendation for comparability, not a
re-derivation of that number** — and §3.7 itself called that slice descriptive rather than a risk
definition. Nothing here narrows the export: all four levels remain equally defensible, and the
other bases stay in the file.

### 3.6 Positive rate by year — and why a chronological split is not exchangeable

Share of days flagged per calendar year, at the default level, under the `any` rule.

In [ ]:
positive_rate = pd.DataFrame(
    {f"{d} · {b}": day_rules(hourly_crossings(residual, thr, d), thr)[0]
     for (d, b), thr in THRESHOLDS.items()}
)
by_year = positive_rate.groupby(positive_rate.index.year).mean()
by_year.index.name = "year"

days_per_year = positive_rate.groupby(positive_rate.index.year).size()
partial = days_per_year[days_per_year < 365]

display(
    by_year.style.format("{:.1%}", na_rep="—")
    .set_caption(f"Share of days flagged, `any` rule, p = {P_DEFAULT:.1%}")
)
print(f"days covered per year: {days_per_year.to_dict()}")
if len(partial):
    print(f"PARTIAL year(s): {partial.to_dict()} — rates for these are not comparable to full years")

**Findings — and a warning the modelling spec has to act on.**

**The low direction's base rate is strongly non-stationary.** Under the `static` and `zero` bases
it is **exactly empty for the record's first four years** and then climbs steeply, ending an order
of magnitude higher than it began. This is not a thresholding artefact: negative residual-load
hours simply did not occur in Germany before 2023, and their share has risen every year since. A
basis with a fixed level therefore does not measure the same thing in 2020 as in 2026.

**The rolling basis is markedly more stable** — the low direction's rate stays within a band a
few-fold wide instead of going from zero to a third of all days. It is *not* stationary either,
and should not be described as such, but it is the only basis under which the early and late
record are remotely comparable.

**The high direction's rate shows no trend under either basis** — it moves around within a band,
which is consistent with `team-EDA.ipynb` §3.7's reading. Nothing here is evidence of a trend in
the high tail, in either direction (§2.3).

**The final year is partial.** The record ends mid-year, so that row covers only part of a year
and is **not comparable to the full years above it**. Because the low phenomenon is summer-heavy
and the high phenomenon winter-heavy, a January-to-September slice *inflates* the low rate and
*deflates* the high one. Read the last row as a fragment, never as the end of a trend.

**The consequence.** A chronological train/test split has **non-exchangeable base rates** for the
low label: any classifier metric computed across it — precision, recall, F1, AUC — is comparing
periods where the label means different things, and is not comparable.

**Recommendation.** Keep the **full record** for training the underlying residual-load regression;
grid load is stable and that history is genuinely useful. But note that the low *label* only
becomes populated partway through the record under any fixed standard. The record is **not
narrowed here** — that decision belongs to the modelling spec, with this table in hand.

### 3.7 How the bases diverge over time

The rolling threshold as a time series per direction, with the static threshold as a horizontal
reference and — for the low direction — zero as a second reference, so the divergence is visible
directly rather than inferred from summary statistics.

The rolling line moves in **steps rather than smoothly**: an extreme quantile of an 8,760-hour
window only shifts when an extreme value enters or leaves that window, which happens on some days
and not others.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for ax, direction in zip(axes, ("high", "low")):
    roll = THRESHOLDS[(direction, "rolling")]
    ax.plot(roll.index, roll.to_numpy(), color=TAIL_COLOR[direction],
            **BASIS_STYLE["rolling"], label=f"rolling ({WINDOW.days}-day trailing)")
    ax.axhline(THRESHOLDS[(direction, "static")].iloc[0], color=COLORS["muted"],
               **BASIS_STYLE["static"], label="static (whole record)")
    if direction == "low":
        ax.axhline(0, color=COLORS["black"], **BASIS_STYLE["zero"],
                   label="zero (physical oversupply boundary)")
    style_timeseries(
        ax,
        f"{DIRECTION_LABEL[direction]} — threshold at p = {P_DEFAULT:.1%}",
        "Residual load threshold (MWh)",
    )
    ax.legend(frameon=False, loc="best")

fig.tight_layout()
plt.show()

for direction in ("high", "low"):
    roll = THRESHOLDS[(direction, "rolling")].dropna()
    print(f"{direction:4} rolling threshold drift: range {roll.max() - roll.min():>9,.0f} MWh "
          f"(min {roll.min():>9,.0f}, max {roll.max():>9,.0f}, std {roll.std():>8,.0f})")

**Findings.**

**The low threshold drifts several times more than the high threshold** — exactly as §6.5's
finding predicts, now in threshold terms rather than as an adjective. The high line wanders inside
a relatively narrow band with no sustained direction. The low line marches steadily downward
across the record and **crosses zero partway through**: for the early years the 1 % low threshold
sits well *above* zero, meaning the bottom 1 % of hours were still comfortably positive and the
`zero` basis flagged nothing at all; by the end of the record it sits well below zero.

That crossing is the single most important thing on this figure. It is why the `zero` and
`static` bases are empty for the early record and why their base rates in §3.6 look the way they
do — and it is the clearest visual argument for the ranking in §3.4. The high direction's static
line is a reasonable summary of its rolling line; the low direction's static line is a poor
summary of anything, sitting far below the rolling threshold for most of the record and far above
it at the end.

---

## 4 Definition 1 — the day-level flag

*"This might be a high-risk day"*, with a reason.

One row per **local `Europe/Berlin` calendar date** — the same day boundary `DERIVED["date"]`
already uses, **not** a rolling 24-hour window. For each day the frame carries its residual-load
maximum and minimum, the hour each occurred, and the day's observation count.

Two day rules are computed for **every** direction × basis. They are **both kept, not chosen
between**:

- **`any`** — the day is flagged if at least one observation crosses the threshold.
- **`3h`** — the day is flagged only if the crossing **persists for at least 3 hours**,
  consecutively.

The `any` rule is sensitive to single-hour excursions, while an operationally stressful event is a
block of hours. Keeping both lets the team see how much of the day-level signal is single-hour
noise — §4.3 measures exactly that.

In [ ]:
daily = pd.DataFrame(index=DAYS)
daily.index.name = "date"

daily["observations"] = obs_per_day
daily["day_complete"] = day_complete
daily["residual_max"] = residual.groupby(DAY).max()
daily["residual_min"] = residual.groupby(DAY).min()
daily["residual_max_hour"] = residual.groupby(DAY).idxmax()
daily["residual_min_hour"] = residual.groupby(DAY).idxmin()

print(f"{len(daily):,} calendar days, {daily['observations'].sum():,} observations "
      f"(= {len(time_series):,} rows in time_series)")
display(daily.head(3))

### 4.1 Flags, and what is deliberately *not* `False`

For each direction × basis × rule: `high_risk = day_max >= high_threshold` and
`low_risk = day_min <= low_threshold`, with the persistence condition additionally required under
the `3h` rule. (Testing the day's max against the threshold is the same thing as asking whether
*any* observation crossed it — the self-check in §7 verifies the two agree.)

**Two situations produce `NaN`, never `False`:**

1. **The rolling threshold is undefined** — the day falls in the record's first 365 days, so there
   is no full trailing year to threshold against.
2. **The day is materially incomplete** — fewer than 23 of its expected 24 observations. A day
   must not be flagged, or cleared, on the strength of a handful of surviving hours after some
   future re-fetch.

`False` means "evaluated, and not at risk". `NaN` means "not evaluable". Collapsing the second
into the first would quietly manufacture thousands of negative labels.

The threshold values themselves are attached to the frame here as well: they are what makes the
export self-describing, and what lets the identical flag be applied later to a *different*
residual-load series (§6.2).

In [ ]:
DAY_RULE_NAMES = ["any", "3h"]

for (direction, basis), thr in THRESHOLDS.items():
    hourly = hourly_crossings(residual, thr, direction)
    any_rule, persist_rule = day_rules(hourly, thr)
    daily[f"{direction}_risk_{basis}_any"] = any_rule
    daily[f"{direction}_risk_{basis}_3h"] = persist_rule
    daily[f"{direction}_threshold_{basis}"] = thr

flag_columns = [
    f"{d}_risk_{b}_{r}" for (d, b) in THRESHOLDS for r in DAY_RULE_NAMES
]
threshold_columns = [f"{d}_threshold_{b}" for (d, b) in THRESHOLDS]

print(f"{len(flag_columns)} flag columns, {len(threshold_columns)} threshold columns")
print("naming convention: {direction}_risk_{basis}_{rule}")
for c in flag_columns:
    print(f"  {c}")

### 4.2 How many days are flagged, and how much the bases disagree

In [ ]:
rows = []
for (direction, basis) in THRESHOLDS:
    for rule in DAY_RULE_NAMES:
        flags = daily[f"{direction}_risk_{basis}_{rule}"]
        rows.append({
            "direction": direction,
            "basis": basis,
            "rule": rule,
            "days flagged": int(flags.sum()),
            "days evaluated": int(flags.notna().sum()),
            "share of evaluated": f"{flags.mean():.2%}",
            "not evaluable": int(flags.isna().sum()),
        })

flag_summary = pd.DataFrame(rows)
display(flag_summary.set_index(["direction", "basis", "rule"]))

# A day flagged in BOTH directions is allowed, never prevented — and reported as a sanity check.
print("days flagged in both directions on the same basis (`any` rule):")
for basis in ("rolling", "static"):
    pair = daily[[f"high_risk_{basis}_any", f"low_risk_{basis}_any"]].dropna()
    n_both = int((pair.iloc[:, 0].astype(bool) & pair.iloc[:, 1].astype(bool)).sum())
    print(f"  {basis:7} : {n_both} of {len(pair):,} evaluable days")

**Findings.**

**The bases disagree far more in the low direction than in the high direction**, which is §3.7's
divergence expressed as day counts. In the high direction the rolling and static bases flag a
broadly similar *share* of the days they evaluate. In the low direction the three bases spread
much wider apart.

**In the low direction the ordering is the opposite of what "zero is the strict physical
boundary" suggests.** The static low threshold sits *below* zero, so the static basis is the
**strictest** of the three — a day has to go further negative to qualify — and the zero basis is
stricter than rolling. The rolling basis flags the most days of the three.

But strictness is not the interesting difference. **The interesting difference is *when* each
basis flags.** Rolling spreads its flags across the whole evaluable record. Zero and static flag
**nothing at all before 2023** and concentrate everything into the final years (§3.6), because
they are fixed while the phenomenon they measure is moving. That is the same fact §3.6 reported as
a base rate and §3.7 drew as a crossing line, now counted in days.

**The `3h` rule removes a large minority of flagged days** in every direction × basis
combination, and never adds one. That containment is structural, not an accident of this data, and
§7's self-check asserts it: every `3h` flag implies its `any` counterpart.

**The both-directions count is zero.** The spec expects this to be "approximately never" at the
default percentile, and it is exactly never, on both bases where the comparison exists: the two
tails are effectively disjoint at the day level. A day would have to hold both a midday oversupply
trough deep enough to clear the 1 % low threshold *and* an evening peak high enough to clear the
1 % high threshold. Physically conceivable; absent from this record. **A non-trivial count here
would be a signal that something is wrong — a threshold or join error — not a finding**, so it is
worth re-reading after any re-fetch.

### 4.3 How much of the `any` signal is a single hour?

This is the question the two day rules exist to answer. If most flagged days cross the threshold
for exactly one hour, the `any` rule is largely measuring momentary excursions rather than
operationally stressful blocks.

In [ ]:
for direction in ("high", "low"):
    hourly = hourly_crossings(residual, THRESHOLDS[(direction, "rolling")], direction)
    per_day = hourly.fillna(False).astype(bool).groupby(DAY).sum()
    flagged = per_day[per_day > 0]
    one_hour = int((flagged == 1).sum())

    print(f"{DIRECTION_LABEL[direction]} (rolling basis, p = {P_DEFAULT:.1%})")
    print(f"  flagged days                  : {len(flagged):,}")
    print(f"  with exactly one crossing hour: {one_hour:,} ({one_hour / len(flagged):.1%})")
    print(f"  median crossing hours per day : {flagged.median():.0f}")
    print(f"  longest single day            : {int(flagged.max())} hours")
    print()

**Findings.** Single-hour excursions are a real but minority effect: roughly a **quarter** of
high-direction flagged days and an **eighth** of low-direction ones cross the threshold for
exactly one hour. So the `any` rule is not *dominated* by momentary excursions, but it is
materially inflated by them — which is exactly what the `3h` rule exists to expose, and why both
rules are kept.

Note the asymmetry: **low-direction events run longer than high-direction events.** The median
flagged low day carries more crossing hours than the median flagged high day, and fewer of them
are single-hour. A solar-driven oversupply trough is a broad midday feature; the high tail's
evening peak is sharper. This is the same structural difference `team-EDA.ipynb` §3.8 found from
the other end — the longest consecutive negative run there was 12 hours, "a recurring midday
phenomenon, not a multi-day state".

Neither rule is adopted as *the* rule here. Both stay in the export, and which one to productionise
is listed in §7 as open for the modelling spec.

### 4.4 The "why" — worked examples

Definition 1 asks for a day **and a reason**. For each flagged day-direction the reason is the
triggering observation: the day's extreme value, the hour it occurred, and the threshold it
crossed.

Examples are taken from the recommended `rolling` basis and spread across the record rather than
picked from the top, so they show typical flagged days rather than only the record's worst.

In [ ]:
def threshold_label(direction, basis):
    """Human-readable name of a threshold, e.g. 'rolling P99' or 'zero'."""
    if basis == "zero":
        return "zero"
    pct = (1 - P_DEFAULT) if direction == "high" else P_DEFAULT
    return f"{basis} P{pct * 100:g}"


def explain_day(date, direction, basis):
    """One line stating why `date` is flagged in `direction` under `basis`."""
    row = daily.loc[date]
    if direction == "high":
        value, when, comparison = row["residual_max"], row["residual_max_hour"], "over"
    else:
        value, when, comparison = row["residual_min"], row["residual_min_hour"], "under"
    thr = row[f"{direction}_threshold_{basis}"]
    return (f"{direction} risk: peak {value:,.0f} MWh at {when:%Y-%m-%d %H:%M}, "
            f"{comparison} the {threshold_label(direction, basis)} threshold of {thr:,.0f} MWh")


N_EXAMPLES = 4

# Kept for §5.2, which demonstrates Definition 2 on these same days.
EXAMPLE_DAYS = {}
for direction in ("high", "low"):
    flagged = daily.index[daily[f"{direction}_risk_rolling_any"].fillna(False).astype(bool)]
    EXAMPLE_DAYS[direction] = flagged[np.linspace(0, len(flagged) - 1, N_EXAMPLES).astype(int)]
    print(f"{DIRECTION_LABEL[direction]} — {len(flagged):,} flagged days, {N_EXAMPLES} examples:")
    for date in EXAMPLE_DAYS[direction]:
        print(f"  {explain_day(date, direction, 'rolling')}")
    print()

**Findings.** Each line is self-contained: it names the day, the triggering hour, the value and
the threshold it crossed, so a flagged day can be audited without re-running the notebook.

In aggregate the flagged hours reproduce §3.7's calendar signature — high-direction crossings
cluster in winter evenings, low-direction crossings around midday in spring and summer — which is
a useful confirmation that the flags pick up the phenomena they were designed for, not an
independent finding. §5.3 plots that distribution properly.

Two things in the examples are worth reading carefully, because both are the rolling basis
behaving as designed rather than misbehaving:

- **The early low-direction example crosses a threshold that is still positive.** A "low residual
  load" day in the record's first evaluable year was not an oversupply day in any physical sense —
  only an unusually low one *for its time*. This is the clearest illustration of why the `zero`
  basis is kept alongside rolling: it is the only basis that would call those days what they were,
  ordinary.
- **Not every flagged day fits the aggregate pattern.** The examples are spread evenly across the
  record rather than picked from the top, so they include ordinary flagged days as well as
  archetypal ones — including early low-direction days whose minimum falls outside the midday
  window that dominates later years. The pattern in §5.3 is a strong central tendency, not a rule
  every flagged day obeys.

---

## 5 Definition 2 — the intra-day flag

*"During 17:00–20:00 there may be high risk for the positive extreme."*

Every observation is flagged against the **same threshold used for its day in Definition 1** —
same percentile, same basis. **No separate intra-day threshold is introduced**, so a flagged hour
and a flagged day always mean the same thing at different grains, and §7's self-check can assert
that the two files agree.

The hourly artifact carries the **plain crossing flags only**. The `3h` persistence rule is a
day-level derivation from consecutive runs of those hourly flags, so repeating it as an hourly
column would store the same information twice under a name that invites misreading.

In [ ]:
hourly_flags = pd.DataFrame(index=time_series.index)
hourly_flags.index.name = "timestamp"
hourly_flags["residual_load"] = residual

for (direction, basis), thr in THRESHOLDS.items():
    hourly_flags[f"{direction}_risk_hour_{basis}"] = hourly_crossings(residual, thr, direction)

hourly_flag_columns = [f"{d}_risk_hour_{b}" for (d, b) in THRESHOLDS]

print(f"{len(hourly_flags):,} rows (= len(time_series)), {len(hourly_flag_columns)} flag columns")
for c in hourly_flag_columns:
    flags = hourly_flags[c]
    print(f"  {c:28} {int(flags.sum()):>6,} flagged / {int(flags.notna().sum()):>6,} evaluated")

### 5.1 The flagged-hour time range

For each flagged day-direction, the time range is the span from the **earliest to the latest
flagged observation of that day, inclusive**.

**This is a span, not a set.** Under the `any` rule the range covers any unflagged gaps between
flagged hours within the same day — if a day crosses at 12:00, stays below at 13:00 and crosses
again at 14:00, its range is 12:00–14:00, not two ranges. That choice is stated here because
"time range" could otherwise reasonably be read as requiring contiguity. If the literal, possibly
non-contiguous *set* of flagged hours is needed, it is available directly from
`data/risk_labels_hourly.csv` — the daily file's range is a summary, not the source of truth.

Under the **`3h`** rule the range is taken over the **qualifying run(s) only**: hours that cross
the threshold but belong to a run shorter than 3 hours are excluded from the span entirely.

In [ ]:
def flagged_range(hourly_flag, rule):
    """Earliest and latest flagged observation per day, as a span (see §5.1).

    Under the `3h` rule only observations belonging to a run of at least PERSISTENCE count.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    if rule == "3h":
        run_id = (filled != filled.shift()).cumsum()
        run_len = filled.groupby([DAY, run_id]).transform("size")
        filled = filled & (run_len >= MIN_RUN)

    stamps = pd.Series(filled.index[filled], index=filled.index[filled].normalize())
    grouped = stamps.groupby(level=0)
    return grouped.min().reindex(DAYS), grouped.max().reindex(DAYS)


for (direction, basis) in THRESHOLDS:
    for rule in DAY_RULE_NAMES:
        start, end = flagged_range(hourly_flags[f"{direction}_risk_hour_{basis}"], rule)
        daily[f"{direction}_range_start_{basis}_{rule}"] = start
        daily[f"{direction}_range_end_{basis}_{rule}"] = end

range_columns = [
    f"{d}_range_{edge}_{b}_{r}"
    for (d, b) in THRESHOLDS for r in DAY_RULE_NAMES for edge in ("start", "end")
]
print(f"{len(range_columns)} range columns added to `daily`")

# The range must exist exactly where the day is flagged.
for (direction, basis) in THRESHOLDS:
    for rule in DAY_RULE_NAMES:
        flagged = daily[f"{direction}_risk_{basis}_{rule}"].fillna(False).astype(bool)
        has_range = daily[f"{direction}_range_start_{basis}_{rule}"].notna()
        assert (flagged == has_range).all(), f"range/flag mismatch: {direction} {basis} {rule}"
print("every flagged day has a range, and every range belongs to a flagged day")

### 5.2 Worked examples — the same days as §4.4

Definition 1 said *which* day and *why*. Definition 2 says *when* within that day.

In [ ]:
def describe_window(date, direction, basis, rule):
    """The spec's phrasing: when within the day the risk sits."""
    start = daily.loc[date, f"{direction}_range_start_{basis}_{rule}"]
    end = daily.loc[date, f"{direction}_range_end_{basis}_{rule}"]
    if pd.isna(start):
        return f"    {rule:3} : not flagged under this rule"
    extreme = "positive" if direction == "high" else "negative"
    hours = int(hourly_flags.loc[str(date.date()), f"{direction}_risk_hour_{basis}"].sum())
    return (f"    {rule:3} : during {start:%H:%M}–{end:%H:%M} there may be {direction} risk "
            f"for the {extreme} extreme ({hours} crossing hours)")


for direction in ("high", "low"):
    print(f"{DIRECTION_LABEL[direction]} (rolling basis)")
    for date in EXAMPLE_DAYS[direction]:
        print(f"  {date:%Y-%m-%d} ({DAY_NAMES[date.dayofweek]})")
        for rule in DAY_RULE_NAMES:
            print(describe_window(date, direction, "rolling", rule))
    print()

**Findings.** The `any` and `3h` ranges differ in an informative way. Where a day's crossing is
one continuous block the two ranges coincide. Where the `3h` range is **narrower**, the day had
isolated crossing hours outside its main block that the persistence rule discards. Where a day is
flagged under `any` but **not at all** under `3h`, its entire crossing was shorter than three
hours — exactly the single-hour excursions §4.3 counted.

This is the practical difference between the two rules, and it is why both are exported: `any`
answers "did this day touch the threshold at all", `3h` answers "did this day sustain it".

### 5.3 When during the day do flagged hours fall?

The hour-of-day distribution of flagged observations, per direction, across all bases. Shares are
within each basis, so bases with very different flagged-hour counts remain comparable.

This is a **check against an existing finding**, not a new one: `team-EDA.ipynb` §3.7 already
established the hour-of-day signature of the two extremes. The question here is only whether the
labels this notebook produces reproduce it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, direction in zip(axes, ("high", "low")):
    for basis in BASES[direction]:
        flags = hourly_flags[f"{direction}_risk_hour_{basis}"].fillna(False).astype(bool)
        if not flags.any():
            continue
        share = flags[flags].index.hour
        counts = pd.Series(share).value_counts().reindex(range(24), fill_value=0).sort_index()
        ax.plot(counts.index, counts / counts.sum() * 100,
                label=f"{basis} (n={int(flags.sum()):,})",
                color=TAIL_COLOR[direction], **BASIS_STYLE[basis], marker="o", markersize=3)

    ax.set_title(DIRECTION_LABEL[direction], fontsize=13, pad=10)
    ax.set_xlabel("Hour of day (Europe/Berlin)")
    ax.set_xticks(range(0, 24, 3))
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.legend(frameon=False, fontsize=9)

axes[0].set_ylabel("Share of that basis's flagged hours (%)", color="grey")
fig.suptitle("When flagged hours fall within the day", fontsize=15, y=1.02)
fig.tight_layout()
plt.show()

for direction in ("high", "low"):
    flags = hourly_flags[f"{direction}_risk_hour_rolling"].fillna(False).astype(bool)
    hours = flags[flags].index
    top = pd.Series(hours.hour).value_counts().head(3).index.tolist()
    top_months = pd.Series(hours.month).value_counts().head(3).index.tolist()
    print(f"{direction:4} (rolling): busiest hours {sorted(top)}, busiest months {sorted(top_months)}")

**Findings — the labels agree with `team-EDA.ipynb` §3.7.**

- **High-direction flagged hours are twin-peaked**: a dominant evening peak and a clear secondary
  morning peak, with **nothing at all overnight**. That is the winter demand profile arriving after
  sunset, when solar contributes nothing.
- **Low-direction flagged hours are sharply concentrated around midday**, in a single narrow band
  centred on the solar maximum.
- The busiest months confirm the other half of the signature: **winter for the high direction,
  spring and early summer for the low.**

This is the mirror-image signature §3.7 described, so the labels reproduce the phenomena they were
designed to capture. **It is a consistency check, not an independent finding** — the same data
produced both.

**One difference between the bases is worth reading.** In the high direction all bases trace
essentially the same curve. In the low direction they agree on the midday band, but the **rolling
basis carries a modest overnight shoulder that the fixed bases almost entirely lack** — roughly an
eighth of its flagged hours fall at night, against a low single-digit percentage for `zero` and
`static`. Those are windy nights that were unusually low *for the trailing year they were measured
against*, while being nowhere near actual oversupply. It is the same effect §4.4's early example
showed, visible here as a population rather than an anecdote: **a relative threshold flags relative
events, and the `zero` basis is what distinguishes "low for its time" from "physically in
surplus".**

Otherwise the bases agree closely on *shape* within each direction, which is itself informative:
they disagree substantially on **which days** and **how many** hours are flagged (§4.2), but
barely at all on **when within the day** those hours fall. The time-of-day signature is a property
of the phenomenon, not of the threshold chosen to detect it.

---

## 6 Export

Two artifacts the modelling spec can load directly, so the risk definition is a **file**, not a
paragraph someone has to re-implement.

Both are written to `data/`, which is **gitignored** under the existing `data/*.csv` pattern —
the same convention `smard.csv` and `rebap.csv` already follow. They are regenerated by running
this notebook.

### A deliberate departure from `smard.csv`'s format

These two files use **plain `sep=","`, `decimal="."`, UTF-8** — pandas defaults — **not** the
German Excel format (`sep=";"`, `decimal=","`, `utf-8-sig`) that `smard.csv` uses.

That is deliberate, not an oversight. `smard.csv` and `rebap.csv` carry their formats because they
arrive from external APIs in that shape. These two are **internal project artifacts** with no such
constraint, and nothing downstream expects German formatting for them. A consumer reads them with
a bare `pd.read_csv(path)`.

### 6.1 `risk_labels_daily.csv` — one row per calendar day

| Group | Columns |
|---|---|
| Day extremes | `residual_max`, `residual_max_hour`, `residual_min`, `residual_min_hour` |
| Completeness | `observations`, `day_complete` |
| Thresholds | `{direction}_threshold_{basis}` |
| Flags | `{direction}_risk_{basis}_{rule}` |
| Time ranges | `{direction}_range_{start,end}_{basis}_{rule}` |

The flag naming convention is stated once and followed exactly, so the file is **self-describing
without the notebook**: `low_risk_rolling_3h` is the low direction, rolling basis, 3-hour
persistence rule.

An empty flag cell means **not evaluable** — either the rolling threshold was undefined (first 365
days) or the day was materially incomplete. It does not mean "not at risk". `day_complete` and
`observations` are carried so a consumer can tell the two apart without recomputing anything.

The rolling threshold columns vary by row; the static and zero ones are constant. All are included
for the same reason — see §6.2.

In [ ]:
DAILY_PATH = DATA_DIR / "risk_labels_daily.csv"
HOURLY_PATH = DATA_DIR / "risk_labels_hourly.csv"

daily_export = daily[
    ["observations", "day_complete",
     "residual_max", "residual_max_hour", "residual_min", "residual_min_hour"]
    + threshold_columns + flag_columns + range_columns
]

daily_export.to_csv(DAILY_PATH)  # pandas defaults: sep=",", decimal="."

print(f"wrote {DAILY_PATH}")
print(f"  {len(daily_export):,} rows x {daily_export.shape[1]} columns "
      f"({DAILY_PATH.stat().st_size / 1024:,.0f} KiB)")
print(f"  one row per calendar day: {daily_export.index.min():%Y-%m-%d} .. "
      f"{daily_export.index.max():%Y-%m-%d}")
display(daily_export.head(3))

### 6.2 Why the thresholds are exported, not just the flags

**Requirement: the exported threshold columns must be sufficient to apply the identical flag to a
*different* residual-load series, by joining on date, with no need to re-run this notebook.**

This is the whole reason the thresholds are in the file. The already-planned forecast-benchmarking
spec has to answer "would this day have been flagged *using SMARD's published day-ahead residual
load* instead of the actual one?" — and the only way that comparison is meaningful is if **exactly
the same thresholds** are applied to both series. A re-run of this notebook against a different
input column would silently produce different rolling thresholds, and the comparison would measure
the thresholds rather than the forecast.

So a later spec joins these threshold columns onto its own series on `date` and applies the same
two comparisons — `value >= {direction}_threshold_{basis}` for high, `<=` for low — plus the same
persistence rule for `3h`. No threshold is recomputed.

**That comparison is out of scope here, and deliberately not performed.** Every threshold and flag
in this notebook is computed on **actual `residual_load` only**; none of the `fc_*` forecast
columns is used anywhere to build a threshold or a flag. This section exports what that future
spec needs and stops there.

### 6.3 `risk_labels_hourly.csv` — one row per observation

Timestamp, `residual_load`, and the hourly crossing flags per direction × basis.

**No threshold values are repeated here.** They are in the daily file; join on date if the
threshold value is needed alongside an hour. Repeating a constant down 67,000 rows would make the
file larger and create two places for the same number to disagree.

The `3h` rule does not appear as an hourly column either — it is a day-level derivation from
consecutive runs of these flags, and belongs in the daily file where it is computed.

In [ ]:
hourly_export = hourly_flags[["residual_load"] + hourly_flag_columns]
hourly_export.to_csv(HOURLY_PATH)  # pandas defaults: sep=",", decimal="."

print(f"wrote {HOURLY_PATH}")
print(f"  {len(hourly_export):,} rows x {hourly_export.shape[1]} columns "
      f"({HOURLY_PATH.stat().st_size / 1024:,.0f} KiB)")
print(f"  one row per observation in time_series: {len(hourly_export) == len(time_series)}")
display(hourly_export.head(3))

### 6.4 Round-trip check

Both files are read back with a bare `pd.read_csv` — no delimiter, no encoding, no decimal
argument — to confirm the format claim in §6 is true rather than merely intended, and that the
grains survive the round trip.

In [ ]:
daily_back = pd.read_csv(DAILY_PATH, index_col="date", parse_dates=["date"])
hourly_back = pd.read_csv(HOURLY_PATH, index_col="timestamp", parse_dates=["timestamp"])

assert len(daily_back) == len(DAYS), f"{len(daily_back)} != {len(DAYS)}"
assert len(hourly_back) == len(time_series), f"{len(hourly_back)} != {len(time_series)}"
assert list(daily_back.columns) == list(daily_export.columns)
assert list(hourly_back.columns) == list(hourly_export.columns)

# Numbers survived the round trip as numbers, not as text.
assert pd.api.types.is_float_dtype(hourly_back["residual_load"])
assert np.allclose(hourly_back["residual_load"], residual.to_numpy())

# Flags survived as three states: True, False, and empty (= not evaluable).
reloaded_flags = daily_back[flag_columns[0]]
print(f"round trip OK — read with a bare pd.read_csv(), no format arguments")
print(f"  daily : {len(daily_back):,} rows x {daily_back.shape[1]} columns")
print(f"  hourly: {len(hourly_back):,} rows x {hourly_back.shape[1]} columns")
print(f"  '{flag_columns[0]}' reloads as "
      f"{reloaded_flags.notna().sum():,} evaluated / {reloaded_flags.isna().sum():,} empty")

---

## 7 Self-check

Run against the **exported files as they sit on disk**, not against the in-memory frames, so the
artifacts a consumer will actually load are what gets verified. Structural only — no hardcoded row
count, date bound or MWh figure, because the record's extent is expected to change.

In [ ]:
def as_nullable_bool(series):
    """Re-typed flags after a CSV round trip: True / False / pd.NA."""
    return series.map({True: True, False: False, "True": True, "False": False}).astype("boolean")


daily_check = pd.read_csv(DAILY_PATH, index_col="date", parse_dates=["date"])
hourly_check = pd.read_csv(HOURLY_PATH, index_col="timestamp", parse_dates=["timestamp"])

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))


# --- grain ------------------------------------------------------------------
check("daily: one row per calendar day in time_series",
      len(daily_check) == time_series["date"].nunique() and daily_check.index.equals(DAYS))
check("daily: index is unique and sorted",
      daily_check.index.is_unique and daily_check.index.is_monotonic_increasing)
check("hourly: exactly len(time_series) rows",
      len(hourly_check) == len(time_series))
check("hourly: index matches time_series exactly",
      hourly_check.index.equals(time_series.index))

# --- time_series was never mutated ------------------------------------------
check("time_series unchanged since loading",
      LOADED == {"rows": len(time_series), "start": time_series.index.min(),
                 "end": time_series.index.max()})

# --- thresholds: constant where they should be, varying where they should ----
for (direction, basis) in THRESHOLDS:
    col = daily_check[f"{direction}_threshold_{basis}"]
    if basis == "rolling":
        check(f"threshold varies by day: {direction}_threshold_{basis}", col.nunique() > 1)
    else:
        check(f"threshold constant across all rows: {direction}_threshold_{basis}",
              col.nunique() == 1)

# --- daily and hourly agree, both ways --------------------------------------
for (direction, basis) in THRESHOLDS:
    day_any = as_nullable_bool(daily_check[f"{direction}_risk_{basis}_any"])
    day_3h = as_nullable_bool(daily_check[f"{direction}_risk_{basis}_3h"])
    hour_flag = as_nullable_bool(hourly_check[f"{direction}_risk_hour_{basis}"])

    hour_day = hour_flag.index.normalize()
    filled = hour_flag.fillna(False).astype(bool)
    hours_per_day = filled.groupby(hour_day).sum().reindex(daily_check.index, fill_value=0)

    evaluated = day_any.notna()
    # every `any`-flagged day has at least one flagged hour, and vice versa
    check(f"{direction}/{basis}: day `any` <-> at least one flagged hour",
          (day_any[evaluated].astype(bool) == (hours_per_day[evaluated] > 0)).all())

    # every `3h`-flagged day has a qualifying consecutive run in the hourly file
    run_id = (filled != filled.shift()).cumsum()
    run_len = filled.groupby([hour_day, run_id]).transform("size").where(filled, 0)
    longest = run_len.groupby(hour_day).max().reindex(daily_check.index, fill_value=0)
    check(f"{direction}/{basis}: day `3h` <-> a run of >= {MIN_RUN} consecutive flagged hours",
          (day_3h[evaluated].astype(bool) == (longest[evaluated] >= MIN_RUN)).all())

    # `3h` implies `any`
    check(f"{direction}/{basis}: every `3h` flag implies its `any` flag",
          not (day_3h[evaluated].astype(bool) & ~day_any[evaluated].astype(bool)).any())

# --- flags are three-state, never silently False ----------------------------
rolling_flags = [c for c in flag_columns if "_rolling_" in c]
check("rolling flags carry empty cells for the days without a full trailing year",
      all(as_nullable_bool(daily_check[c]).isna().sum() == int((~day_complete).sum())
          + (len(DAYS) - int(THRESHOLDS[("high", "rolling")].notna().sum()))
          for c in rolling_flags))

failed = [label for label, ok in checks if not ok]
for label, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
print()
if failed:
    raise AssertionError(f"{len(failed)} self-check(s) failed: {failed}")
print(f"all {len(checks)} self-checks passed")

---

## 8 Findings

### What this notebook decided

**The two extremes of residual load are treated as two directions, not one signed target.**
This answers the open question `team-EDA.ipynb` closed on. The justification is §3.7's
mirror-image calendar signature — recent, summer, midday, weekend for the low tail; spread across
years, winter, weekday, evening for the high tail — and §5.3 confirms the labels reproduce it.
Whether a *model* should be fitted as one two-sided target or two separate ones is a modelling
decision and stays open.

### Recommended bases and level

| | Recommendation |
|---|---|
| **Label bases** | `rolling` for both directions; `zero` additionally for the low direction |
| **Demoted** | `static` — computed, exported and plotted as a reference, but **not recommended as a label basis** |
| **Level** | **1 %**, for continuity with §3.7's 1 % tail-by-rank framing |

**Why `rolling`.** It is the only **causal** basis: day `D`'s threshold uses only `[D-365, D-1]`,
so the label could have been produced the morning of that day and a day-ahead model can be
evaluated against it honestly. It is also the only basis that keeps the positive rate remotely
comparable across a chronological split.

**Why `zero` as well.** It is the only basis that means anything **outside this dataset**. Below
zero, wind and solar alone exceed national demand and the surplus must be exported, curtailed or
paid away. It takes no parameter and does not move when the record is re-fetched. The high
direction has **no equivalent physical anchor** in region-`DE` data, so the asymmetry between the
directions is deliberate.

**Why `static` is demoted.** It is **unstable under re-fetch** — a property of the record's extent
rather than of the grid, and widening the fetch back to 2019 has already moved it once — and it is
**not causal**, judging 2022 days against quantiles containing 2025–2026 data. Its costs are not
hypothetical: it is the basis under which the low label is empty for four straight years.

**What `rolling` costs.** It is undefined for the record's first 365 days, and it moves as history
accumulates, so identical physics can be labelled differently in different periods. That is the
price of a stable base rate, and the ranking accepts it.

### How the bases diverge

**In the high direction they barely diverge at all** — the rolling threshold wanders around the
static one with no sustained direction, and the two flag a similar share of days.

**In the low direction they diverge enormously.** The low rolling threshold drifts roughly three
times as far as the high one across the record and **crosses zero partway through**: early in the
record the bottom 1 % of hours were still comfortably positive, and by the end they are firmly
negative. This is §6.5's "the distribution is stretching downward while P90 barely moves",
expressed as thresholds rather than as an adjective.

### The base-rate warning

**The low direction's base rate is strongly non-stationary.** Under the `static` and `zero` bases
it is **exactly empty for the record's first four years** and ends an order of magnitude higher —
because negative residual-load hours did not occur in Germany before 2023 and their share has
risen every year since. The `rolling` basis is markedly more stable but is **not stationary
either**.

**Consequence for the modelling spec:** a chronological train/test split has **non-exchangeable
base rates** for the low label. Any classifier metric computed across it — precision, recall, F1,
AUC — compares periods in which the label means different things, and is not comparable.

**Recommendation:** keep the **full record** for training the underlying residual-load regression;
grid load is stable and that history is useful. But treat the low label's base rate as moving. The
record is **not narrowed here** — that decision belongs to the modelling spec, with §3.6's table
in hand.

### The limits of this label — restated, because they travel with it

1. **No margin.** The high direction is a *relative* extreme — the highest hours in this record —
   **not** a demonstrated approach to a capacity limit. Region-`DE` data carries no
   installed-capacity or availability figures, so "tight reserve margins" is not a claim this
   label supports.
2. **No regional detail.** The dominant real driver of German redispatch is north–south
   transmission congestion, which can occur at perfectly moderate *national* residual load and is
   invisible here. **This label is a national-balance proxy and must be described as one wherever
   it is presented.**
3. **No intervention record.** Nothing in `data/smard.csv` records whether a TSO actually
   intervened. There is no ground truth here. The label is a **plausibility proxy, never a
   validated outcome.**

And the framing: this is a **level-based magnitude definition, not a probability-of-intervention
model**. It produces the candidate label a future model would be trained to predict.

### Open for the modelling spec

- **One target or two?** This notebook establishes the two directions are different phenomena. It
  does not decide whether to fit one two-sided model, two independent models, or a single
  regression on `residual_load` with the flags applied afterwards. The last of these is attractive
  precisely because the thresholds are exported separately from the flags.
- **Which day rule to productionise.** `any` and `3h` are both exported. `any` answers "did this
  day touch the threshold", `3h` answers "did it sustain it". Roughly a quarter of high-direction
  `any` days are single-hour excursions, so the choice materially changes the label.
- **Ramp magnitude as a third input.** `team-EDA.ipynb` §6.4 found that residual load's *rate of
  change* is growing faster than its level, and that level extremes and ramp extremes are "not the
  same hours". This notebook's flags are **level-based only**. Whether a ramp-based stress mode
  belongs in the risk definition is explicitly left open.
- **Capacity normalisation.** Thresholds here are in absolute MWh. Once installed-capacity data is
  imported, a future spec may need to express them per unit of capacity instead.
- **The percentile is a recommendation, not a finding.** All four candidate levels remain
  defensible; §3.5's table is there so the team can overrule the 1 % default.